In [3]:
import numpy as np

# ------------------ 激活函数及其导数 ------------------
def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(z):
    """Sigmoid 的导数，输入为 z（线性输出）"""
    s = sigmoid(z)
    return s * (1 - s)

# ------------------ 损失函数：二分类交叉熵 ------------------
def compute_loss(y_true, y_pred):
    """
    计算二分类交叉熵损失，y_pred 为网络输出（概率）
    损失 = - [y*log(p) + (1-y)*log(1-p)] 的平均
    """
    # 避免 log(0) 造成数值问题
    y_pred = np.clip(y_pred, 1e-12, 1 - 1e-12)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# ------------------ 神经网络类 ------------------
class NeuralNetwork:
    def __init__(self, layer_sizes):
        """
        layer_sizes: 列表，如 [2, 4, 1] 表示输入层2个神经元，1个隐藏层4个神经元，输出层1个神经元
        随机初始化权重和偏置
        """
        self.num_layers = len(layer_sizes) - 1  # 权重矩阵的个数
        self.weights = []
        self.biases = []
        for i in range(self.num_layers):
            # 用小的随机数初始化，避免对称
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * 0.1
            b = np.random.randn(layer_sizes[i+1], 1) * 0.1
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, x):
        """
        前向传播
        x: 输入数据，形状 (特征数, 样本数)  或 (特征数, 1) 单个样本
        返回每层的 (z, a)，以便反向传播使用
        """
        a = x
        activations = [a]  # 存储 a[0] 到 a[L]
        zs = []            # 存储 z[1] 到 z[L]
        for w, b in zip(self.weights, self.biases):
            z = np.dot(w, a) + b   # 线性变换
            zs.append(z)
            a = sigmoid(z)         # 激活（所有层均使用 Sigmoid）
            activations.append(a)
        return activations, zs

    def backward(self, x, y, activations, zs, learning_rate):
        """
        反向传播，计算梯度并更新参数
        x, y 形状: (特征数, 样本数)
        activations, zs: 前向传播的中间结果
        """
        m = x.shape[1]  # 样本数

        # 输出层误差 (Sigmoid + 交叉熵的优雅形式)
        delta = activations[-1] - y  # 形状 (n_L, m)

        # 遍历每一层，从输出层反向传播到第一个隐藏层
        for l in range(self.num_layers - 1, -1, -1):
            # 计算权重和偏置的梯度（对当前层）
            # dW = (1/m) * delta * a[l]^T
            dW = (1 / m) * np.dot(delta, activations[l].T)
            db = (1 / m) * np.sum(delta, axis=1, keepdims=True)

            # 更新参数（梯度下降）
            self.weights[l] -= learning_rate * dW
            self.biases[l]  -= learning_rate * db

            # 如果不是输入层，则继续计算前一层的误差
            if l > 0:
                # delta[l-1] = (W[l]^T * delta[l]) ⊙ sigmoid'(z[l-1])
                delta = np.dot(self.weights[l].T, delta) * sigmoid_prime(zs[l-1])

    def train(self, X, Y, epochs, learning_rate, verbose=True):
        """
        训练网络
        X: 输入数据 (特征数, 样本数)
        Y: 真实标签 (输出维度, 样本数)
        epochs: 迭代次数
        learning_rate: 学习率
        """
        for epoch in range(epochs):
            # 前向传播
            activations, zs = self.forward(X)
            # 计算损失
            loss = compute_loss(Y, activations[-1])
            # 反向传播与参数更新
            self.backward(X, Y, activations, zs, learning_rate)
            if verbose and epoch % 200 == 0:
                print(f"Epoch {epoch:4d}, Loss: {loss:.6f}")

    def predict(self, X):
        """返回预测概率"""
        activations, _ = self.forward(X)
        return activations[-1]

# ------------------ 示例：XOR 问题 ------------------
if __name__ == "__main__":
    # XOR 输入和标签 (每个样本是一列)
    X = np.array([[0, 0, 1, 1],
                  [0, 1, 0, 1]])   # 形状 (2, 4)
    Y = np.array([[0, 1, 1, 0]])   # 形状 (1, 4)

    # 创建网络：输入2维，隐藏层4个神经元，输出1维（二分类）
    nn = NeuralNetwork([2, 4, 1])

    print("开始训练 XOR 问题...")
    nn.train(X, Y, epochs=2000, learning_rate=0.5)

    # 查看预测结果
    pred = nn.predict(X)
    print("\n训练后的预测概率：")
    for i in range(4):
        print(f"输入 {X[:,i]} -> 预测 {pred[0,i]:.4f} (真实 {Y[0,i]})")
    print("\n分类结果（阈值0.5）：", (pred > 0.5).astype(int))

开始训练 XOR 问题...
Epoch    0, Loss: 0.693881
Epoch  200, Loss: 0.693139
Epoch  400, Loss: 0.693138
Epoch  600, Loss: 0.693135
Epoch  800, Loss: 0.693133
Epoch 1000, Loss: 0.693130
Epoch 1200, Loss: 0.693125
Epoch 1400, Loss: 0.693120
Epoch 1600, Loss: 0.693112
Epoch 1800, Loss: 0.693100

训练后的预测概率：
输入 [0 0] -> 预测 0.5002 (真实 0)
输入 [0 1] -> 预测 0.4962 (真实 1)
输入 [1 0] -> 预测 0.5037 (真实 1)
输入 [1 1] -> 预测 0.4996 (真实 0)

分类结果（阈值0.5）： [[1 0 1 0]]


In [4]:
import sys

def verify_pytorch():
    # 1. 检查 PyTorch 是否已安装
    try:
        import torch
    except ImportError:
        print("❌ 错误：未检测到 PyTorch，请先安装 PyTorch。")
        sys.exit(1)

    print("=" * 60)
    print("🔥 PyTorch 环境深度检测报告")
    print("=" * 60)
    
    # 2. 基础版本信息
    print(f"✅ PyTorch 版本: {torch.__version__}")
    print(f"✅ Python 版本: {sys.version.split()[0]}")
    
    # 3. 硬件加速器检测 (CUDA / MPS)
    device = "cpu"
    print("-" * 60)
    
    # 检测 NVIDIA GPU (CUDA)
    cuda_available = torch.cuda.is_available()
    print(f"🖥️ CUDA (NVIDIA GPU) 可用: {cuda_available}")
    if cuda_available:
        device = "cuda"
        gpu_count = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"   ├─ CUDA 驱动版本: {torch.version.cuda}")
        print(f"   ├─ cuDNN 版本: {torch.backends.cudnn.version()}")
        print(f"   ├─ GPU 数量: {gpu_count}")
        print(f"   ├─ 默认 GPU: {gpu_name}")
        print(f"   └─ 显存大小: {gpu_mem:.2f} GB")
    else:
        print("   └─ (未检测到 NVIDIA GPU 或未正确安装 GPU 版 PyTorch)")
        
    # 检测 Apple Silicon (MPS)
    mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    print(f"🍎 MPS (Apple Silicon) 可用: {mps_available}")
    if mps_available and device == "cpu":
        device = "mps"
        
    print(f"🎯 当前默认推理设备: {device.upper()}")
    print("-" * 60)

    # 4. 基础计算与张量测试
    print("⏳ 正在进行张量计算测试...")
    try:
        # CPU 测试
        x_cpu = torch.randn(1000, 1000, device="cpu")
        y_cpu = torch.randn(1000, 1000, device="cpu")
        _ = torch.mm(x_cpu, y_cpu)
        print("✅ CPU 矩阵乘法测试: 通过")

        # GPU / MPS 测试
        if device != "cpu":
            # 创建一个在加速器上的张量
            x_dev = torch.randn(1000, 1000, device=device)
            y_dev = torch.randn(1000, 1000, device=device)
            z_dev = torch.mm(x_dev, y_dev)
            print(f"✅ {device.upper()} 矩阵乘法测试: 通过")
            
            # 清理 GPU 缓存
            if device == "cuda":
                torch.cuda.empty_cache()
                
    except Exception as e:
        print(f"❌ 计算测试失败: {e}")
        return

    # 5. 最终结论
    print("=" * 60)
    if device != "cpu":
        print(f"🎉 恭喜！PyTorch 已成功配置并可调用 {device.upper()} 进行加速！")
    else:
        print("⚠️ PyTorch 已安装，但当前仅能使用 CPU 运行。")
        print("   如需 GPU 加速，请检查是否安装了 GPU 版本的 PyTorch 及显卡驱动。")
    print("=" * 60)

if __name__ == "__main__":
    verify_pytorch()

🔥 PyTorch 环境深度检测报告
✅ PyTorch 版本: 2.5.1+cu121
✅ Python 版本: 3.11.6
------------------------------------------------------------
🖥️ CUDA (NVIDIA GPU) 可用: True
   ├─ CUDA 驱动版本: 12.1
   ├─ cuDNN 版本: 90100
   ├─ GPU 数量: 1
   ├─ 默认 GPU: NVIDIA GeForce RTX 4060 Laptop GPU
   └─ 显存大小: 8.00 GB
🍎 MPS (Apple Silicon) 可用: False
🎯 当前默认推理设备: CUDA
------------------------------------------------------------
⏳ 正在进行张量计算测试...
✅ CPU 矩阵乘法测试: 通过
✅ CUDA 矩阵乘法测试: 通过
🎉 恭喜！PyTorch 已成功配置并可调用 CUDA 进行加速！


In [10]:
import torch 

a = torch.randn(3,3)
b = torch.randn(3,3)
print("=== 矩阵 A (CPU) ===")
print(a)
print("\n=== 矩阵 B (CPU) ===")
print(b)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n当前使用设备: {device}")

a_gpu = a.to(device)
b_gpu = b.to(device)

c_gpu = a_gpu @ b_gpu
print(f"\n结果所在设备: {c_gpu.device}")
print("\n=== 矩阵 C = A × B (GPU) ===")
print(c_gpu)

c_cpu = c_gpu.cpu()
print(f"\n转回 CPU 后设备: {c_cpu.device}")
print("\n=== 矩阵 C (CPU) ===")
print(c_cpu)

=== 矩阵 A (CPU) ===
tensor([[ 0.7068,  0.4244,  1.6519],
        [ 0.6235, -0.6902,  0.4269],
        [ 0.5336,  0.8908,  0.6498]])

=== 矩阵 B (CPU) ===
tensor([[ 0.2666, -2.0241,  1.1464],
        [-2.4842, -0.2305, -0.3418],
        [-1.6142, -0.6246, -0.9712]])

当前使用设备: cuda

结果所在设备: cuda:0

=== 矩阵 C = A × B (GPU) ===
tensor([[-3.5323, -2.5603, -0.9390],
        [ 1.1918, -1.3696,  0.5361],
        [-3.1197, -1.6913, -0.3239]], device='cuda:0')

转回 CPU 后设备: cpu

=== 矩阵 C (CPU) ===
tensor([[-3.5323, -2.5603, -0.9390],
        [ 1.1918, -1.3696,  0.5361],
        [-3.1197, -1.6913, -0.3239]])


In [18]:
import torch
w = torch.tensor(3.0,requires_grad = True)

print(f'w = {w.item()}')
print(f'w.requires_grad = {w.requires_grad}')

y = w ** 3 + 2 * w

print(f'\ny = w³ + 2w = {y.item()}')

y.backward()

print(f'dy/dw (PyTorch自动求导) = {w.grad.item()}')

w_val =3.0
y_manual = w_val ** 3 + 2 * w_val
dy_dw_manual = 3 * w_val ** 2 + 2
print(f"\n--- 手算验证 ---")
print(f"y    (手算) = {y_manual}")
print(f"dy/dw (手算) = {dy_dw_manual}")

assert abs(w.grad.item() - dy_dw_manual) < 1e-6,'梯度验证失败！'
print("\n✅ PyTorch 自动求导结果与手算结果完全一致！")

w = 3.0
w.requires_grad = True

y = w³ + 2w = 33.0
dy/dw (PyTorch自动求导) = 29.0

--- 手算验证 ---
y    (手算) = 33.0
dy/dw (手算) = 29.0

✅ PyTorch 自动求导结果与手算结果完全一致！


In [21]:
import torch
w_values = [1.0,2.0,3.0,-5.0,-6.0,-7.0]

print(f"{'w':>6} | {'y (auto)':>10} | {'dy/dw (auto)':>14} | {'dy/dw (手算)':>14} | {'匹配':>4}")
print("-" * 65)

for val in w_values:
    w = torch.tensor(val,requires_grad=True)
    y = w ** 3 + 2 * w
    y.backward()
    
    dy_manual = 3 * val ** 2 + 2
    match = "✅" if abs(w.grad.item() - dy_manual) < 1e-6 else "❌"
    print(f"{val:>6.1f} | {y.item():>10.1f} | {w.grad.item():>14.1f} | {dy_manual:>14.1f} | {match:>4}")

     w |   y (auto) |   dy/dw (auto) |     dy/dw (手算) |   匹配
-----------------------------------------------------------------
   1.0 |        3.0 |            5.0 |            5.0 |    ✅
   2.0 |       12.0 |           14.0 |           14.0 |    ✅
   3.0 |       33.0 |           29.0 |           29.0 |    ✅
  -5.0 |     -135.0 |           77.0 |           77.0 |    ✅
  -6.0 |     -228.0 |          110.0 |          110.0 |    ✅
  -7.0 |     -357.0 |          149.0 |          149.0 |    ✅


In [ ]:
import torch
#设置超参数：学习率和训练轮数
LEARNING_RATE = 0.01
NUM_EPOCHS = 1000
#真实的权重和偏置
TRUE_W1 = 2.5
TRUE_W2 = -1.3
TRUE_B = 4.0
#生成数据
x = torch.linspace(-5,5,100).reshape(-1,1).to(torch.float32)
noise = torch.randn(100,1) * 2.0
y_true = TRUE_W1 * (x ** 2) + TRUE_W2 * x +TRUE_B + noise
#设备转移
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️  当前训练设备: {device}\n")
#直接在目标设备上创建参数
w1 = torch.randn(1,requires_grad=True,device=device)
w2 = torch.randn(1,requires_grad=True,device=device)
b = torch.randn(1,requires_grad=True,device=device)

#数据转移到设备
x,y_true = x.to(device),y_true.to(device)


#训练循环
for epoch in range(NUM_EPOCHS):
#前向传播
    y_pred = w1 * (x ** 2) + w2 * x + b
#计算损失
    loss = ((y_pred - y_true) ** 2).mean()
#反向传播：自动求导
    loss.backward()
#更新参数
    with torch.no_grad():
       w1.data -= LEARNING_RATE * w1.grad
       w2.data -= LEARNING_RATE * w2.grad
       b.data -= LEARNING_RATE * b.grad
#梯度清零
       w1.grad.zero_()
       w2.grad.zero_()
       b.grad.zero_()
#每200轮打印一次Loss:       
    if (epoch + 1) % 200 == 0:
        print(f"Epoch [{epoch+1:04d}/{NUM_EPOCHS}] | Loss: {loss.item():.4f}")
#转回cpu并对比
w1_final = w1.cpu().item()
w2_final = w2.cpu().item()
b_final = b.cpu().item()

print("\n" + "="*40)
print("🔍 破解结果对比：")
print(f"{'参数':<6} | {'真实值 (Ground Truth)':<20} | {'模型拟合值':<15}")
print("-" * 45)
print(f"{'w1':<6} | {TRUE_W1:<22.4f} | {w1_final:<15.4f}")
print(f"{'w2':<6} | {TRUE_W2:<22.4f} | {w2_final:<15.4f}")
print(f"{'b':<6} | {TRUE_B:<22.4f} | {b_final:<15.4f}")
print("="*40)

⚙️  当前训练设备: cuda

Epoch [0200/1000] | Loss: nan
Epoch [0400/1000] | Loss: nan
Epoch [0600/1000] | Loss: nan
Epoch [0800/1000] | Loss: nan
Epoch [1000/1000] | Loss: nan

🔍 破解结果对比：
参数     | 真实值 (Ground Truth)   | 模型拟合值          
---------------------------------------------
w1     | 2.5000                 | nan            
w2     | -1.3000                | nan            
b      | 4.0000                 | nan            
